In [ ]:
!pip install -q "numpy<2.0.0"

In [ ]:
!pip install -q --force-reinstall torch vllm

In [ ]:
!pip install -q evaluate bert_score scipy
!pip install -q datasets pandas
!pip install -q pyarrow
!pip install -q pinecone sentence-transformers
!pip install -q transformers accelerate bitsandbytes pandas tqdm



In [ ]:
!pip install -q -U transformers huggingface_hub vllm

In [ ]:
!pip install -q sacrebleu


In [ ]:
from datasets import load_dataset
import pandas as pd

print("Authenticating and loading FLORES+ dataset...")


HF_TOKEN = "INSERT THE TOKEn"


eng_data = load_dataset("openlanguagedata/flores_plus", "eng_Latn", split="devtest", token=HF_TOKEN)
ben_data = load_dataset("openlanguagedata/flores_plus", "ben_Beng", split="devtest", token=HF_TOKEN)

print("Aligning the translations...")

df_baseline = pd.DataFrame({
    "id": eng_data["id"],
    "english_text": eng_data["text"],
    "human_bengali_reference": ben_data["text"]
})

csv_filename = "flores200_human_baseline.csv"
df_baseline.to_csv(csv_filename, index=False, encoding="utf-8-sig")

print(f"Success! Loaded {len(df_baseline)} human-translated pairs.")
df_baseline.head()

In [ ]:
import sys
import io
import os

# Ensure the stable V0 engine is used
os.environ["VLLM_USE_V1"] = "0"


try:
    sys.stdout.fileno()
except io.UnsupportedOperation:
    sys.stdout.fileno = lambda: 1
    sys.stderr.fileno = lambda: 2

import pandas as pd
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

print("Loading baseline dataset...")
df = pd.read_csv("flores200_human_baseline.csv")

model_name = "Qwen/Qwen2.5-7B-Instruct-AWQ"

print("Initializing Colab-Optimized vLLM engine...")
llm = LLM(
    model=model_name,
    quantization="awq",
    max_model_len=1024,           # Reduced from 2048 to save massive VRAM
    gpu_memory_utilization=0.8,   # Leave 20% of the GPU free for the actual math
    enforce_eager=True,           # Disable memory-hogging CUDA graphs
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

sampling_params = SamplingParams(temperature=0.0, max_tokens=256)

print("Preparing prompts...")
prompts = []
for eng_text in df['english_text']:
    prompt_text = (
        "Translate the following English text into standard Bengali. "
        "Do not use transliteration. Only output the Bengali translation.\n\n"
        f"English: {eng_text}\nBengali:"
    )

    messages = [
        {"role": "system", "content": "You are an expert English-to-Bengali translator."},
        {"role": "user", "content": prompt_text}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    prompts.append(formatted_prompt)

print("Executing high-speed batch translation...")
outputs = llm.generate(prompts, sampling_params)

translations = [output.outputs[0].text.strip() for output in outputs]

df['machine_bengali_output'] = translations
output_filename = "flores200_qwen_baseline_translations.csv"
df.to_csv(output_filename, index=False, encoding="utf-8-sig")

print(f"Translation complete! Saved to {output_filename}")

In [ ]:
import pandas as pd
import evaluate

print("Initializing metrics...")
bertscore = evaluate.load("bertscore")
chrf = evaluate.load("chrf")

# Load your freshly generated dataset
df = pd.read_csv("flores200_qwen_baseline_translations.csv")

refs = df['human_bengali_reference'].tolist()
preds = df['machine_bengali_output'].tolist()

# chrF++ expects references as a list of lists
formatted_refs = [[ref] for ref in refs]

print("Computing chrF++ score...")
chrf_result = chrf.compute(predictions=preds, references=formatted_refs, word_order=2)
print(f"🏆 System Baseline chrF++ Score: {chrf_result['score']:.2f}")

print("Computing BERTScore (this checks semantic meaning)...")
bert_results = bertscore.compute(
    predictions=preds,
    references=refs,
    lang="bn",
    model_type="bert-base-multilingual-cased"
)

df['bert_f1_score'] = bert_results['f1']
mean_bert = sum(bert_results['f1']) / len(bert_results['f1'])
print(f"System Baseline Mean BERTScore (F1): {mean_bert:.4f}")

# Save the final evaluated file
output_file = "flores200_evaluated_baseline.csv"
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"\nEvaluation complete! Detailed breakdown saved to {output_file}")

In [ ]:
from datasets import load_dataset
import pandas as pd
import re

print("Downloading English-Bengali corpus from OPUS-100...")
# The language pair is listed alphabetically on Hugging Face as 'bn-en'
dataset = load_dataset("Helsinki-NLP/opus-100", "bn-en", split="train")

# Extract the English and Bengali texts into a Pandas DataFrame
df = pd.DataFrame({
    "english_text": [item['en'] for item in dataset['translation']],
    "human_bengali_reference": [item['bn'] for item in dataset['translation']]
})

print(f"Total sentences loaded: {len(df)}")

# Define the clinical keyword filter
# I have populated this with standard medical terminology to capture health-related text
medical_keywords = [
    "hospital", "disease", "patient", "doctor", "treatment",
    "surgery", "vaccine", "blood", "cancer", "medicine",
    "health", "clinical", "virus", "infection", "syndrome",
    "dose", "symptom", "therapy", "diagnosis", "medical", "diabetes"
]

print("Filtering for medical terminology...")
# Create a regex pattern to match any of the keywords exactly (case-insensitive)
pattern = '|'.join([rf'\b{word}\b' for word in medical_keywords])
medical_df = df[df['english_text'].str.contains(pattern, flags=re.IGNORECASE, na=False)]

# Drop any accidental duplicates to ensure strict academic diversity
medical_df = medical_df.drop_duplicates(subset=['english_text'])

print(f"Total medical sentences found: {len(medical_df)}")

# Randomly sample exactly 1,000 sentences for the final benchmark
if len(medical_df) >= 1000:
    final_df = medical_df.sample(n=1000, random_state=42).reset_index(drop=True)
else:
    print("Warning: Found fewer than 1,000 sentences. Using all available.")
    final_df = medical_df.reset_index(drop=True)

# Add an ID column and clean up the structure
final_df['id'] = final_df.index + 1
final_df = final_df[['id', 'english_text', 'human_bengali_reference']]

# Save the dataset to be used in your RAG pipeline
output_filename = "my_medical_evaluation_dataset.csv"
final_df.to_csv(output_filename, index=False, encoding="utf-8-sig")

print(f" Success! Saved {len(final_df)} medical sentence pairs to {output_filename}")
final_df.head()

# MY RAG PIPELINE

In [ ]:
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

print("Connecting to Pinecone...")
# Replace with your actual API key
PINECONE_API_KEY = "INSERT THE API KEY"
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medical-rag-thesis"

# Create a serverless index if it doesn't exist
if index_name not in pc.list_indexes().names():
    print(f"Creating new index: {index_name}...")
    pc.create_index(
        name=index_name,
        dimension=384, # This must match the embedding model's output size
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

print("Loading embedding model onto CPU to protect GPU VRAM...")
# all-MiniLM-L6-v2 is a blazing fast, tiny model perfect for this
embedder = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

print("Uploading medical terminology to the cloud...")
# I am using a mock dictionary for demonstration.
medical_dictionary = [
    {"id": "term_1", "english": "Hypertension", "bengali": "উচ্চ রক্তচাপ (Uchcha raktachap)"},
    {"id": "term_2", "english": "Myocardial Infarction", "bengali": "হৃদরোগের আক্রমণ (Hridroger akromon) / হার্ট অ্যাটাক"},
    {"id": "term_3", "english": "Tuberculosis", "bengali": "যক্ষ্মা (Jakhma)"},
    {"id": "term_4", "english": "Diabetes Mellitus", "bengali": "বহুমূত্র রোগ (Bohumutro rog) / ডায়াবেটিস"}
]

# Convert the English terms into math (vectors) and upload the Bengali translations as metadata
vectors_to_upsert = []
for item in medical_dictionary:
    vector = embedder.encode(item["english"]).tolist()
    vectors_to_upsert.append({
        "id": item["id"],
        "values": vector,
        "metadata": {"english_term": item["english"], "bengali_term": item["bengali"]}
    })

index.upsert(vectors=vectors_to_upsert)
print("Upload complete! Your vector database is live.")

In [ ]:
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

print("Connecting to Pinecone...")


pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medical-rag-thesis"

# Create the serverless cloud index if it doesn't already exist
if index_name not in pc.list_indexes().names():
    print(f"Creating new cloud database: {index_name}...")
    pc.create_index(
        name=index_name,
        dimension=384, # The exact mathematical dimension of our embedding model
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

print("Loading the embedding model onto the CPU...")
embedder = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

print("Formatting the clinical glossary...")
# The custom glossary tailored to your dataset
medical_glossary = [
    {"id": "t1", "en": "hospital", "bn": "হাসপাতাল (Hospital)"},
    {"id": "t2", "en": "disease", "bn": "রোগ (Rog)"},
    {"id": "t3", "en": "patient", "bn": "রোগী (Rogi)"},
    {"id": "t4", "en": "doctor", "bn": "ডাক্তার / চিকিৎসক (Daktar / Chikitsok)"},
    {"id": "t5", "en": "treatment", "bn": "চিকিৎসা (Chikitsa)"},
    {"id": "t6", "en": "surgery", "bn": "অস্ত্রোপচার (Ostropachar)"},
    {"id": "t7", "en": "vaccine", "bn": "টিকা / ভ্যাকসিন (Tika / Vaccine)"},
    {"id": "t8", "en": "blood", "bn": "রক্ত (Rokto)"},
    {"id": "t9", "en": "cancer", "bn": "ক্যান্সার / কর্কটরোগ (Cancer / Korkotrog)"},
    {"id": "t10", "en": "medicine", "bn": "ওষুধ / ঔষধ (Oshudh)"},
    {"id": "t11", "en": "health", "bn": "স্বাস্থ্য (Shasthyo)"},
    {"id": "t12", "en": "clinical", "bn": "ক্লিনিক্যাল / চিকিৎসাগত (Clinical / Chikitsagot)"},
    {"id": "t13", "en": "virus", "bn": "ভাইরাস (Virus)"},
    {"id": "t14", "en": "infection", "bn": "সংক্রমণ (Songkromon)"},
    {"id": "t15", "en": "syndrome", "bn": "সিনড্রোম / লক্ষণসমষ্টি (Syndrome)"},
    {"id": "t16", "en": "dose", "bn": "মাত্রা / ডোজ (Matra / Dose)"},
    {"id": "t17", "en": "symptom", "bn": "লক্ষণ / উপসর্গ (Lokkhon / Uposorgo)"},
    {"id": "t18", "en": "therapy", "bn": "থেরাপি / চিকিৎসা পদ্ধতি (Therapy)"},
    {"id": "t19", "en": "diagnosis", "bn": "রোগ নির্ণয় (Rog Nirnoy)"},
    {"id": "t20", "en": "diabetes", "bn": "ডায়াবেটিস / বহুমূত্র রোগ (Diabetes / Bohumutro rog)"},
    {"id": "t21", "en": "blood pressure", "bn": "রক্তচাপ (Roktochap)"},
    {"id": "t22", "en": "heart attack", "bn": "হার্ট অ্যাটাক / হৃদরোগের আক্রমণ (Heart Attack)"},
    {"id": "t23", "en": "tuberculosis", "bn": "যক্ষ্মা (Jakhma)"},
    {"id": "t24", "en": "stroke", "bn": "স্ট্রোক (Stroke)"},
    {"id": "t25", "en": "asthma", "bn": "হাঁপানি / অ্যাজমা (Hapani)"},
    {"id": "t26", "en": "fever", "bn": "জ্বর (Jwor)"},
    {"id": "t27", "en": "pain", "bn": "ব্যথা / যন্ত্রণা (Betha)"},
    {"id": "t28", "en": "pharmacy", "bn": "ফার্মেসি / ঔষধের দোকান (Pharmacy)"},
    {"id": "t29", "en": "nurse", "bn": "নার্স / সেবিকা (Nurse / Sebika)"},
    {"id": "t30", "en": "blood test", "bn": "রক্ত পরীক্ষা (Rokto porikkha)"}
]

print("Uploading vectors to Pinecone...")
vectors_to_upsert = []
for item in medical_glossary:
    # Convert the English word into mathematical vectors
    vector = embedder.encode(item["en"]).tolist()

    # Package it with the Bengali translation as metadata
    vectors_to_upsert.append({
        "id": item["id"],
        "values": vector,
        "metadata": {"english_term": item["en"], "bengali_term": item["bn"]}
    })

# Upload everything to the cloud
index.upsert(vectors=vectors_to_upsert)

print(" Success! Your RAG database is populated and ready to be queried.")

In [ ]:
import sys
import io
import os
import pandas as pd
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer

# --- 1. Colab Patches ---
os.environ["VLLM_USE_V1"] = "0"
try:
    sys.stdout.fileno()
except io.UnsupportedOperation:
    sys.stdout.fileno = lambda: 1
    sys.stderr.fileno = lambda: 2

# --- 2. Initialize vLLM FIRST! ---
# We must start vLLM before any other PyTorch models wake up to prevent multiprocessing crashes
print("Initializing Colab-Optimized vLLM engine FIRST...")
model_name = "Qwen/Qwen2.5-7B-Instruct-AWQ"
llm = LLM(
    model=model_name,
    quantization="awq",
    max_model_len=1024,
    gpu_memory_utilization=0.8,
    enforce_eager=True,
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
sampling_params = SamplingParams(temperature=0.0, max_tokens=256)

# --- 3. Initialize Pinecone & CPU Embedder SECOND ---
print("Connecting to Pinecone and loading embedder...")
# Paste your actual Pinecone API key here

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index("medical-rag-thesis")

embedder = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

# --- 4. The Live Retrieval Function ---
def retrieve_medical_context(english_sentence):
    query_vector = embedder.encode(english_sentence).tolist()
    search_results = index.query(
        vector=query_vector,
        top_k=2,
        include_metadata=True
    )

    context_pieces = []
    for match in search_results['matches']:
        eng = match['metadata']['english_term']
        ben = match['metadata']['bengali_term']
        context_pieces.append(f"'{eng}' translates to '{ben}'")

    if not context_pieces:
        return "No specific medical context found."

    return "\n".join(context_pieces)

# --- 5. Run the Augmented Pipeline ---
print("Loading medical dataset...")
df = pd.read_csv("your_medical_evaluation_dataset.csv")

print("Retrieving context and preparing prompts...")
prompts = []

for eng_text in df['english_text']:
    retrieved_context = retrieve_medical_context(eng_text)

    prompt_text = (
        "You are an expert English-to-Bengali medical translator. "
        "Translate the following English text into standard Bengali. "
        "Use the provided Medical Glossary Context to ensure clinical accuracy.\n\n"
        f"Medical Glossary Context:\n{retrieved_context}\n\n"
        f"English: {eng_text}\nBengali:"
    )

    messages = [
        {"role": "system", "content": "You are a clinical translation assistant."},
        {"role": "user", "content": prompt_text}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    prompts.append(formatted_prompt)

print("Executing high-speed RAG batch translation...")
outputs = llm.generate(prompts, sampling_params)
translations = [output.outputs[0].text.strip() for output in outputs]

df['machine_bengali_output'] = translations
output_filename = "rag_augmented_translations.csv"
df.to_csv(output_filename, index=False, encoding="utf-8-sig")

print(f"RAG Translation complete! Saved to {output_filename}")

In [ ]:
import pandas as pd
import evaluate

print("Initializing metrics...")
bertscore = evaluate.load("bertscore")
chrf = evaluate.load("chrf")

print("Loading RAG-augmented dataset...")
df = pd.read_csv("rag_augmented_translations.csv")

refs = df['human_bengali_reference'].tolist()
preds = df['machine_bengali_output'].tolist()

# chrF++ expects references as a list of lists
formatted_refs = [[ref] for ref in refs]

print("Computing RAG chrF++ score...")
chrf_result = chrf.compute(predictions=preds, references=formatted_refs, word_order=2)
print(f" RAG System chrF++ Score: {chrf_result['score']:.2f}")

print("Computing RAG BERTScore (this checks semantic meaning)...")
bert_results = bertscore.compute(
    predictions=preds,
    references=refs,
    lang="bn",
    model_type="bert-base-multilingual-cased",
    batch_size=16
)

df['bert_f1_score'] = bert_results['f1']
mean_bert = sum(bert_results['f1']) / len(bert_results['f1'])
print(f"RAG System Mean BERTScore (F1): {mean_bert:.4f}")

output_file = "rag_evaluated_final.csv"
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"\nFinal evaluation complete! Detailed breakdown saved to {output_file}")

In [ ]:
import pandas as pd
import evaluate

print("Initializing metrics...")
bertscore = evaluate.load("bertscore")
chrf = evaluate.load("chrf")

print("Loading RAG-augmented dataset...")
df = pd.read_csv("rag_augmented_translations.csv")

refs = df['human_bengali_reference'].tolist()
preds = df['machine_bengali_output'].tolist()

# chrF++ expects references as a list of lists
formatted_refs = [[ref] for ref in refs]

print("Computing RAG chrF++ score...")
chrf_result = chrf.compute(predictions=preds, references=formatted_refs, word_order=2)
print(f"RAG System chrF++ Score: {chrf_result['score']:.2f}")

print("Computing RAG BERTScore (this checks semantic meaning)...")
bert_results = bertscore.compute(
    predictions=preds,
    references=refs,
    lang="bn",
    model_type="bert-base-multilingual-cased",
    batch_size=16
)

df['bert_f1_score'] = bert_results['f1']
mean_bert = sum(bert_results['f1']) / len(bert_results['f1'])
print(f" RAG System Mean BERTScore (F1): {mean_bert:.4f}")

output_file = "rag_evaluated_final.csv"
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"\nFinal evaluation complete! Detailed breakdown saved to {output_file}")

**IMPROVING THE SCORE**

In [ ]:
import sys
import io
import os
import pandas as pd
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

# Force the stable V0 engine
os.environ["VLLM_USE_V1"] = "0"

# Apply the Colab Monkey Patch
try:
    sys.stdout.fileno()
except io.UnsupportedOperation:
    sys.stdout.fileno = lambda: 1
    sys.stderr.fileno = lambda: 2

print("Loading the hard medical evaluation dataset...")
df = pd.read_csv("your_medical_evaluation_dataset.csv")

model_name = "Qwen/Qwen2.5-7B-Instruct-AWQ"

print("Initializing Colab-Optimized vLLM engine...")
llm = LLM(
    model=model_name,
    quantization="awq",
    max_model_len=1024,
    gpu_memory_utilization=0.8,
    enforce_eager=True,
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
sampling_params = SamplingParams(temperature=0.0, max_tokens=256)

print("Preparing strictly unassisted prompts...")
prompts = []
for eng_text in df['english_text']:
    prompt_text = (
        "Translate the following English text into standard Bengali. "
        "Do not use transliteration. Only output the Bengali translation.\n\n"
        f"English: {eng_text}\nBengali:"
    )

    messages = [
        {"role": "system", "content": "You are an expert English-to-Bengali translator."},
        {"role": "user", "content": prompt_text}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    prompts.append(formatted_prompt)

print("Executing high-speed batch translation (Unassisted)...")
outputs = llm.generate(prompts, sampling_params)

translations = [output.outputs[0].text.strip() for output in outputs]

df['machine_bengali_output'] = translations
output_filename = "medical_unassisted_baseline_translations.csv"
df.to_csv(output_filename, index=False, encoding="utf-8-sig")

print(f"Unassisted translation complete! Saved to {output_filename}")

In [ ]:
import pandas as pd
import evaluate

print("Initializing metrics...")
bertscore = evaluate.load("bertscore")
chrf = evaluate.load("chrf")

print("Loading medical_unassisted_baseline_translations...")
df = pd.read_csv("medical_unassisted_baseline_translations.csv")

refs = df['human_bengali_reference'].tolist()
preds = df['machine_bengali_output'].tolist()

# chrF++ expects references as a list of lists
formatted_refs = [[ref] for ref in refs]

print("Computing RAG chrF++ score...")
chrf_result = chrf.compute(predictions=preds, references=formatted_refs, word_order=2)
print(f"RAG System chrF++ Score: {chrf_result['score']:.2f}")

print("Computing RAG BERTScore (this checks semantic meaning)...")
bert_results = bertscore.compute(
    predictions=preds,
    references=refs,
    lang="bn",
    model_type="bert-base-multilingual-cased",
    batch_size=16
)

df['bert_f1_score'] = bert_results['f1']
mean_bert = sum(bert_results['f1']) / len(bert_results['f1'])
print(f" RAG System Mean BERTScore (F1): {mean_bert:.4f}")

output_file = "rag_evaluated_final.csv"
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"\nFinal evaluation complete! Detailed breakdown saved to {output_file}")

In [ ]:
import pandas as pd
from scipy.stats import pearsonr
import sacrebleu

print("Loading the evaluated dataset...")

df = pd.read_csv("rag_evaluated_final.csv")

if 'bert_f1_score' not in df.columns:
    print("Error: 'bert_f1_score' not found. Please ensure the evaluation script ran correctly.")
else:
    bert_scores = df['bert_f1_score'].tolist()

    print("Calculating sentence-level chrF++ scores...")
    chrf_scores = []

    # Calculate chrF++ for each individual sentence to compare against its BERT F1 score
    for index, row in df.iterrows():
        ref = [str(row['human_bengali_reference'])]
        pred = str(row['machine_bengali_output'])

        # sacrebleu.sentence_chrf extracts the float score for the single sentence
        score = sacrebleu.sentence_chrf(pred, ref).score
        chrf_scores.append(score)

    df['sentence_chrf_score'] = chrf_scores

    print("Calculating Pearson Correlation...")
    # This computes the correlation coefficient (r) and the p-value
    correlation, p_value = pearsonr(bert_scores, chrf_scores)

    print("\n" + "="*50)
    print(f"Pearson Correlation (BERT F1 vs chrF++): {correlation:.4f}")
    print(f"p-value: {p_value:.4e}")
    print("="*50 + "\n")

    if p_value < 0.05:
        print("CONCLUSION: The correlation is statistically significant.")
    else:
        print("CONCLUSION: The correlation is not statistically significant.")

    output_file = "medical_baseline_with_pearson.csv"
    df.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"\nFinal dataset with all sentence-level metrics saved to {output_file}")